# SatellaRhythm-Edge v0.3 — final *research release candidate*

**Portable single-lead ECG (not PPG) • real-data training • rhythm-first fine-tuning • dual heads • ONNX edge export • transparent release gates**

This notebook addresses v0.2R's main observed failures: AF overprediction (96% recall but ~44% precision), declining `OTHER` recognition, and weak transfer of signal-quality labels from 12-lead ECGs to wearable lead I. Its goal is to **test** whether a staged training schedule and gentler loss balancing produce a better compromise, not to guarantee that a final safe medical model exists.

**Data**: PhysioNet Challenge 2017 (short single-lead AliveCor ECG) for three rhythms plus real `~` uninterpretable recordings; PhysioNet Challenge 2011 Set A for *auxiliary* quality supervision from lead I (the ground truth concerns the entire 12-lead recording). MIT-BIH AFDB training-only rhythm extension is optional and **off by default**. No synthetic noisy samples are generated.

**Key design safeguards**

- Fixed, label-stratified record-disjoint training/validation/test partitions; deduplication before splitting; patient-disjoint status **not verified**.
- *Stage 1*: learn N / AF / Other on true rhythm-labelled 2017 ECGs. *Stage 2*: joint training with reduced signal-quality contribution; two predefined recipes are compared **on validation only**.
- Mild stratified sampling **without multiplying by inverse-frequency class weights again**. This avoids the double correction in v0.2R.
- Record-level multi-window evaluation, including precision/recall per class, false-NORMAL rate, noisy ECG detection, usable-ECG coverage, and source-specific results.
- Abstention/rejection thresholds set using validation data. Within-run final test evaluated only **after** model and policy are frozen; the underlying 2017 dataset has been examined in our earlier versions, so this is **not** a new independent external holdout.
- ONNX two-head export, parity test, local CPU benchmark, standalone inference script and detailed model card. **No automatic publishing or raw ECGs in the package.**

**Clinical limitation**: research prototype only. Must **never** be used for medical diagnosis, screening, triage, treatment or reassurance. A missed AF or other abnormal rhythm can cause harm. Passing the notebook's research thresholds would **not** establish clinical readiness.

Source links: [2017](https://physionet.org/content/challenge-2017/1.0.0/), [2011](https://physionet.org/content/challenge-2011/1.0.0/), [AFDB](https://physionet.org/content/afdb/1.0.0/). Review original licences and attribution obligations for each dataset.


## 1. Install dependencies — run this once in a fresh GPU Colab

Choose **Runtime → Change runtime type → A100 or L4 GPU**. Archive downloads (~GB scale) can take longer than training. The installation does not replace Colab's installed PyTorch.


In [ ]:
import subprocess, sys
packages = ['wfdb>=4.1,<6', 'onnx>=1.16,<2', 'onnxruntime>=1.18,<2',
            'safetensors>=0.4,<1', 'scipy>=1.11,<2', 'scikit-learn>=1.3,<2',
            'pandas>=2,<3','requests>=2.30,<3','tqdm>=4.66,<5','matplotlib>=3.7,<4']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Dependencies installed; PyTorch:', __import__('torch').__version__)


## 2. Configuration, seeds and hardware

You can run the notebook as-is. `ENABLE_AFDB=False` deliberately keeps additional AFDB windows out of the core comparison: introducing only positive AF windows from another device may bias the model toward AF again. If you later enable it, **that is a new experiment** and must not be described as the default v0.3 result.


In [ ]:
import os, io, csv, re, json, time, math, random, zipfile, tarfile, hashlib, shutil, copy, platform, itertools
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext
import requests, wfdb
import numpy as np, pandas as pd
import scipy.signal as sps
from scipy.io import loadmat
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, f1_score, recall_score,
    balanced_accuracy_score, precision_score, roc_auc_score)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

SEED=2026
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=Path('/content/satellarhythm_edge_v03') if Path('/content').exists() else Path('./satellarhythm_edge_v03')
DATA=ROOT/'data';MODEL_DIR=ROOT/'model';DATA.mkdir(parents=True,exist_ok=True);MODEL_DIR.mkdir(parents=True,exist_ok=True)
FS=300;SECONDS=10;WINDOW_SEC=SECONDS;WINDOW=FS*SECONDS
RHYTHM=['NORMAL','ATRIAL_FIBRILLATION','OTHER_RHYTHM']
QUALITY=['INTERPRETABLE','UNINTERPRETABLE']
RHYTHM_FROM_2017={'N':0,'A':1,'O':2,'~':-100}
QUALITY_FROM_2017={'N':0,'A':0,'O':0,'~':1}
WARM_EPOCHS=9; JOINT_EPOCHS=12; WARM_PATIENCE=4; JOINT_PATIENCE=4
LR_WARM=5e-4; LR_JOINT=1.8e-4; WEIGHT_DECAY=1e-3
BATCH17=48 if DEVICE.type=='cuda' else 12; BATCH11=16 if DEVICE.type=='cuda' else 8
EVAL_WINDOWS_2017=5; EVAL_WINDOWS_2011=1
ENABLE_AFDB=False; AFDB_MAX_RECORDS=4;AFDB_AF_WINDOWS_PER_RECORD=40
# Two predeclared joint recipes; validation chooses between them, never test data.
RECIPES=[{'name':'quality_gentle','q2017':.30,'q2011':.07},
         {'name':'quality_moderate','q2017':.44,'q2011':.13}]
print({'device':str(DEVICE),'torch':torch.__version__,'model_output':str(MODEL_DIR),
       'afdb_enabled':ENABLE_AFDB,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})


## 3. Download PhysioNet 2017 and 2011 archives

Uses the exact working direct-download archives from v0.2R. Cached files are reused. A failed or truncated download stops execution; it never silently produces an incomplete training set.


In [ ]:
URL2017='https://physionet.org/files/challenge-2017/1.0.0/training2017.zip'
LABEL_URL2017='https://physionet.org/files/challenge-2017/1.0.0/REFERENCE-v3.csv'
URL2011='https://physionet.org/files/challenge-2011/1.0.0/set-a.tar.gz'
ZIP17=DATA/'training2017.zip'; TAR11=DATA/'set-a.tar.gz'

def download_cached(url,path,min_bytes=1000):
    if path.exists() and path.stat().st_size>=min_bytes:
        print('Cached:',path.name,round(path.stat().st_size/1e6,1),'MB');return path
    temp=path.with_suffix(path.suffix+'.part')
    try:
        with requests.get(url,stream=True,timeout=(25,180),headers={'User-Agent':'SatellaRhythmResearch/0.2R'}) as resp:
            resp.raise_for_status(); length=int(resp.headers.get('Content-Length',0))
            with temp.open('wb') as f:
                for block in tqdm(resp.iter_content(chunk_size=1<<20),total=math.ceil(length/(1<<20)) if length else None,desc=path.name):
                    if block:f.write(block)
        if temp.stat().st_size<min_bytes:raise RuntimeError(f'{path.name} was truncated: {temp.stat().st_size} bytes')
        os.replace(temp,path)
    finally:
        if temp.exists():temp.unlink()
    return path

download_cached(URL2017,ZIP17,50_000_000)
download_cached(URL2011,TAR11,50_000_000)
assert zipfile.is_zipfile(ZIP17),'2017 ZIP invalid'
assert tarfile.is_tarfile(TAR11),'2011 tar archive invalid'
print('Both original datasets downloaded successfully.')

## 4. Load, deduplicate and audit the portable 2017 ECGs

Use **v3 revised annotations**, which have substantially more recordings labelled `~` than the first version of the challenge's training labels. Deduplicate exact waveform copies *before* splitting.


In [ ]:
with zipfile.ZipFile(ZIP17) as zf:
    mats={Path(n).stem:n for n in zf.namelist() if n.lower().endswith('.mat')}
    try:
        resp=requests.get(LABEL_URL2017,timeout=45);resp.raise_for_status()
        txt=resp.text;label_source=LABEL_URL2017
    except Exception as exc:
        candidates=[n for n in zf.namelist() if Path(n).name.upper() in ('REFERENCE-V3.CSV','REFERENCE.CSV')]
        if not candidates:raise RuntimeError('Missing v3 labels; cannot safely continue') from exc
        txt=zf.read(candidates[0]).decode('utf-8-sig');label_source='ZIP:'+candidates[0]
    labels={row[0].strip():row[1].strip() for row in csv.reader(io.StringIO(txt)) if len(row)>=2 and row[1].strip() in RHYTHM_FROM_2017}
    assert len(labels)>=8000,f'2017 labels unexpectedly incomplete: {len(labels)}'
    rec17=[];seen=set();invalid=[];dup=[]
    for rid,label in tqdm(labels.items(),desc='Read 2017 ECGs'):
        try:
            raw=np.asarray(loadmat(io.BytesIO(zf.read(mats[rid])))['val'],dtype=np.float32).reshape(-1)
            if len(raw)<FS*8 or not np.isfinite(raw).all() or np.ptp(raw)<1e-6:raise ValueError('bad wave')
            digest=hashlib.sha256(raw.tobytes()).hexdigest()
            if digest in seen:dup.append(rid);continue
            seen.add(digest)
            rec17.append({'id':'c2017/'+rid,'source':'c2017','x':raw,'ry':RHYTHM_FROM_2017[label],
                          'qy':QUALITY_FROM_2017[label],'source_label':label})
        except Exception as exc:invalid.append((rid,type(exc).__name__))
print('2017 usable:',len(rec17),'duplicate:',len(dup),'invalid:',len(invalid),'labels:',Counter(r['source_label'] for r in rec17))
assert len(rec17)>8000 and sum(r['qy']==1 for r in rec17)>25,'2017 loading/labels unexpectedly incomplete'

## 5. Load real 2011 Set A signal-quality labels

**Important label caveat:** these are whole-12-lead judgements. The notebook extracts only lead I and never invents a 2011 rhythm label. Lead-I quality may disagree with the original multi-lead judgement.


In [ ]:
DIR11=DATA/'challenge2011_seta'; DIR11.mkdir(exist_ok=True)
with tarfile.open(TAR11,'r:gz') as tf:
    members={Path(m.name).name:m for m in tf.getmembers() if m.isfile()}
    def load_list(name):
        if name not in members:raise FileNotFoundError(f'Missing {name} in Set A tar. Files include: {list(members)[-12:]}')
        return set(v.strip() for v in tf.extractfile(members[name]).read().decode('utf-8-sig').splitlines()
                   if re.fullmatch(r'\d{6,10}',v.strip()))
    acceptable=load_list('RECORDS-acceptable'); unacceptable=load_list('RECORDS-unacceptable')
    assert not (acceptable&unacceptable),'Contradictory quality labels in source'
    targets={**{rid:0 for rid in acceptable},**{rid:1 for rid in unacceptable}}
    assert len(targets)>900,f'Only {len(targets)} quality labels found; inspect tar contents.'
    # Never blindly extract tar members. Write only numeric WFDB waveform/header basenames.
    for rid in tqdm(targets,desc='Extract WFDB Set A records'):
        for ext in ('.hea','.dat'):
            name=rid+ext
            if name not in members:continue
            dst=DIR11/name
            if not dst.exists():
                with tf.extractfile(members[name]) as f, dst.open('wb') as g:shutil.copyfileobj(f,g)

rec11=[];failure11=[];hashes11=set()
for rid,qy in tqdm(targets.items(),desc='Load lead I (2011)'):
    try:
        r=wfdb.rdrecord(str(DIR11/rid))
        if 'I' not in r.sig_name:raise ValueError('lead I absent')
        lead=np.asarray(r.p_signal[:,r.sig_name.index('I')],dtype=np.float32)
        raw=sps.resample_poly(lead,FS,int(r.fs)).astype(np.float32)
        if raw.size<FS*8 or not np.isfinite(raw).all() or float(np.ptp(raw))<1e-6:raise ValueError('invalid lead')
        digest=hashlib.sha256(raw.tobytes()).hexdigest()
        if digest in hashes11:continue
        hashes11.add(digest)
        rec11.append({'id':'c2011/'+rid,'source':'c2011','x':raw,'ry':-100,'qy':qy,
                      'source_label':'acceptable' if qy==0 else 'unacceptable'})
    except Exception as exc:failure11.append((rid,str(exc)[:100]))
print('2011 lead-I loaded:',len(rec11),'failures:',len(failure11),'quality:',Counter(r['source_label'] for r in rec11))
if failure11:print('First three failed records (no personal details):',failure11[:3])
assert len(rec11)>850 and len(set(r['qy'] for r in rec11))==2,'2011 data/quality parse failed; inspect before training.'
print('CAUTION: 2011 quality labels are 12-lead record-level judgements applied to lead I for exploratory learning.')

## 6. Immutable record-disjoint splits and source audit

The ECG sources do not supply dependable unique patient identifiers for both datasets. This is **not** a verified patient-disjoint study, despite record-level deduplication. The same seed and split logic are used across all v0.3 candidate recipes.


In [ ]:
def split_records(rows,seed=SEED):
    idx=np.arange(len(rows))
    y=[r['source_label'] for r in rows]
    tr,rest=train_test_split(idx,test_size=.30,stratify=y,random_state=seed)
    va,te=train_test_split(rest,test_size=.50,stratify=np.asarray(y)[rest],random_state=seed)
    return [[rows[i] for i in part] for part in [tr,va,te]]
train17,val17,test17=split_records(rec17)
train11,val11,test11=split_records(rec11)
for rows, tag in [(train17,'2017 train'),(val17,'2017 val'),(test17,'2017 test'),(train11,'2011 train'),(val11,'2011 val'),(test11,'2011 test')]:
    print(tag,len(rows),dict(Counter(r['source_label'] for r in rows)))
train_rows=train17+train11;val_rows=val17+val11;test_rows=test17+test11
for a,b in ((train_rows,val_rows),(train_rows,test_rows),(val_rows,test_rows)):
    assert not ({r['id'] for r in a}&{r['id'] for r in b}),'Record leakage detected'
assert sum(r['source']=='c2017' and r['qy']==1 for r in test_rows)>=5,'Test set missing noisy 2017 ECGs'
print('Train/validation/test recording IDs disjoint; patient-disjoint NOT verified.')

In [ ]:
SPLIT_AUDIT={
    'seed':SEED,'labels_2017':'REFERENCE-v3.csv',
    'recording_split_patient_disjoint_verified':False,
    'c2017':{k:dict(Counter(r['source_label'] for r in rr)) for k,rr in
             [('train',train17),('validation',val17),('test',test17)]},
    'c2011':{k:dict(Counter(r['source_label'] for r in rr)) for k,rr in
             [('train',train11),('validation',val11),('test',test11)]},
}
assert len({r['id'] for r in train_rows+val_rows+test_rows})==len(train_rows+val_rows+test_rows)
print(json.dumps(SPLIT_AUDIT,indent=2))


## 7. Optional AFDB AF rhythm extension (training only)

Default **off**. If enabled, extra windows are taken from annotations explicitly marked AFIB. These are NOT used as `normal` or signal-quality examples, and no same-record AFDB windows are added to the held-out evaluation. This is not an independent AFDB test.


In [ ]:
if ENABLE_AFDB:
    BASE_AF='https://physionet.org/files/afdb/1.0.0'
    AF_DIR=DATA/'afdb';AF_DIR.mkdir(exist_ok=True)
    record_lines=requests.get(BASE_AF+'/RECORDS',timeout=40);record_lines.raise_for_status()
    available=[x.strip() for x in record_lines.text.splitlines() if re.fullmatch(r'\d{5}',x.strip())]
    # Some AFDB annotations-only records have no waveform; skip them explicitly.
    available=[rid for rid in available if rid not in ('00735','03665')]
    added=[]
    for rid in available[:AFDB_MAX_RECORDS]:
        try:
            for ext in ('.hea','.dat','.atr'):
                download_cached(f'{BASE_AF}/{rid}{ext}',AF_DIR/(rid+ext),min_bytes=50)
            annotation=wfdb.rdann(str(AF_DIR/rid),'atr')
            signal=wfdb.rdrecord(str(AF_DIR/rid),channels=[0]).p_signal[:,0].astype(np.float32)
            hz=int(wfdb.rdheader(str(AF_DIR/rid)).fs)
            starts=[(int(s),str(t).strip()) for s,t in zip(annotation.sample,annotation.aux_note) if str(t).strip().startswith('(')]
            windows=[]
            for k,(start,tag) in enumerate(starts):
                stop=starts[k+1][0] if k+1<len(starts) else len(signal)
                if tag.startswith('(AFIB') and stop-start>=WINDOW_SEC*hz:
                    positions=np.arange(start,stop-WINDOW_SEC*hz+1,WINDOW_SEC*hz,dtype=int)
                    if len(positions):windows.extend(positions.tolist())
            # Select spaced points; avoid random nondeterminism on reruns.
            if windows:
                selected=np.asarray(windows,dtype=int)[np.linspace(0,len(windows)-1,min(len(windows),AFDB_AF_WINDOWS_PER_RECORD),dtype=int)]
                for j,start in enumerate(selected):
                    cropped=signal[int(start):int(start)+WINDOW_SEC*hz]
                    x=sps.resample_poly(cropped,FS,hz).astype(np.float32)
                    if len(x)==WINDOW and np.isfinite(x).all() and np.ptp(x)>1e-5:
                        added.append({'id':f'afdb/{rid}/{int(start)}','source':'afdb','x':x,'ry':1,'qy':-100,'source_label':'AFIB'})
            print('AFDB',rid,'AF windows retained:',sum(x['id'].startswith('afdb/'+rid+'/') for x in added))
        except Exception as exc:
            print('AFDB record unavailable:',rid,type(exc).__name__,str(exc)[:120])
    if not added:raise RuntimeError('AFDB enabled, but zero valid annotated AFIB windows loaded. Stop rather than silently continue.')
    train_rows.extend(added)
    print('Extra AFDB TRAINING windows:',len(added),'(no validation/test labels, no quality labels).')
else:
    print('AFDB disabled. Real 2017 + real 2011 data are included by default.')

## 8. Window preprocessing and balanced source-specific loaders

Both tasks see the same 300 Hz, 10-second input representation. Quality-only 2011 samples are drawn from a **separate loader**, preventing their labels from dominating rhythm training. Training windows are real crops from recordings; no synthetic corruption is used.

Unlike v0.2R, there are **no additional inverse-frequency class weights on top of sampling**.


In [ ]:
def preprocess(x):
    a=np.asarray(x,dtype=np.float32).reshape(-1)
    if len(a)!=WINDOW:raise ValueError(f'Expected {WINDOW} ECG samples')
    if not np.isfinite(a).all():raise ValueError('Nonfinite ECG')
    a=a-np.median(a)
    scale=max(float(np.percentile(np.abs(a),90)),1e-6)
    return np.clip(a/scale,-6,6).astype(np.float32)

def crop(raw,start):
    raw=np.asarray(raw,dtype=np.float32).reshape(-1)
    if len(raw)>=WINDOW:return raw[int(start):int(start)+WINDOW]
    if len(raw)<2:raise ValueError('Invalid waveform')
    return np.pad(raw,(0,WINDOW-len(raw)),mode='reflect')[:WINDOW]

class TrainECG(Dataset):
    def __init__(self,rows):self.rows=list(rows)
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i];raw=r['x'];start=np.random.randint(0,max(1,len(raw)-WINDOW+1))
        return torch.from_numpy(preprocess(crop(raw,start))[None,:].copy()),int(r['ry']),int(r['qy'])

class EvalECG(Dataset):
    def __init__(self,rows):
        self.rows=list(rows);self.loc=[]
        for idx,r in enumerate(rows):
            nw=EVAL_WINDOWS_2011 if r['source']=='c2011' else EVAL_WINDOWS_2017
            for start in np.linspace(0,max(0,len(r['x'])-WINDOW),nw,dtype=int):
                self.loc.append((idx,int(start)))
    def __len__(self):return len(self.loc)
    def __getitem__(self,i):
        idx,start=self.loc[i];return torch.from_numpy(preprocess(crop(self.rows[idx]['x'],start))[None,:].copy()),idx

def weighted_loader(rows,group_probs,batch_size,num_samples=None,seed=SEED):
    counter=Counter(group_for_sample(r) for r in rows)
    bad=set(counter)-set(group_probs)
    if bad:raise ValueError(f'Unexpected groups: {bad}')
    probs={k:float(v) for k,v in group_probs.items() if counter.get(k,0)>0}
    total=sum(probs.values());probs={k:v/total for k,v in probs.items()}
    weights=np.asarray([probs[group_for_sample(r)]/counter[group_for_sample(r)] for r in rows],dtype=np.float64)
    gen=torch.Generator().manual_seed(seed)
    sampler=WeightedRandomSampler(torch.tensor(weights,dtype=torch.double),
                                  num_samples=num_samples or len(rows),replacement=True,generator=gen)
    return DataLoader(TrainECG(rows),batch_size=batch_size,sampler=sampler,num_workers=0,
                      pin_memory=DEVICE.type=='cuda')

def group_for_sample(r):
    if r['source']=='c2011':return '2011_bad' if r['qy']==1 else '2011_good'
    if r['source']=='afdb':return 'afdb_AF'
    if r['qy']==1:return '2017_Q'
    return ['2017_N','2017_AF','2017_OTHER'][r['ry']]

WARM_GROUPS={'2017_N':.44,'2017_AF':.18,'2017_OTHER':.38}
JOINT_GROUPS={'2017_N':.37,'2017_AF':.16,'2017_OTHER':.39,'2017_Q':.08}
if ENABLE_AFDB:JOINT_GROUPS={**{k:v*.96 for k,v in JOINT_GROUPS.items()},'afdb_AF':.04}
TRAIN17_RHYTHM=[r for r in train17 if r['ry']>=0]
TRAIN_JOINT=[r for r in train_rows if r['source']!='c2011']
warm_loader=weighted_loader(TRAIN17_RHYTHM,WARM_GROUPS,BATCH17,seed=SEED+11)
val_loader=DataLoader(EvalECG(val_rows),batch_size=BATCH17,shuffle=False,num_workers=0)
test_loader=DataLoader(EvalECG(test_rows),batch_size=BATCH17,shuffle=False,num_workers=0)
print('Training records:',Counter(group_for_sample(r) for r in train_rows))
print('Warm-up source draw:',WARM_GROUPS,'joint draw:',JOINT_GROUPS)
print('Evaluation record count:',len(val_rows),len(test_rows),'window count:',len(val_loader.dataset),len(test_loader.dataset))


## 9. Compact dual-head network

A residual 1D CNN with **mean and maximum pooling** before the heads. Max pooling may help when clinically important patterns are brief rather than distributed over the entire 10-second window. This is an architectural hypothesis, not a validated benefit.


In [ ]:
class ResBlock(nn.Module):
    def __init__(self,inc,outc,stride=1):
        super().__init__()
        self.net=nn.Sequential(nn.Conv1d(inc,outc,7,stride,3,bias=False),nn.BatchNorm1d(outc),nn.SiLU(),
            nn.Conv1d(outc,outc,5,1,2,bias=False),nn.BatchNorm1d(outc))
        self.short=(nn.Identity() if inc==outc and stride==1 else
                    nn.Sequential(nn.Conv1d(inc,outc,1,stride,bias=False),nn.BatchNorm1d(outc)))
        self.act=nn.SiLU()
    def forward(self,x):return self.act(self.net(x)+self.short(x))

class RhythmQualityNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem=nn.Sequential(nn.Conv1d(1,32,15,2,7,bias=False),nn.BatchNorm1d(32),nn.SiLU())
        self.trunk=nn.Sequential(ResBlock(32,64,2),ResBlock(64,64),ResBlock(64,128,2),ResBlock(128,128),
            ResBlock(128,192,2),ResBlock(192,192),ResBlock(192,256,2),ResBlock(256,256))
        self.pool_avg=nn.AdaptiveAvgPool1d(1);self.pool_max=nn.AdaptiveMaxPool1d(1)
        self.project=nn.Sequential(nn.Linear(512,192),nn.SiLU(),nn.Dropout(.15))
        self.rhythm=nn.Linear(192,3);self.quality=nn.Linear(192,2)
    def forward(self,x):
        f=self.trunk(self.stem(x))
        z=torch.cat((self.pool_avg(f).squeeze(-1),self.pool_max(f).squeeze(-1)),1)
        z=self.project(z)
        return self.rhythm(z),self.quality(z)

model=RhythmQualityNet().to(DEVICE)
PARAMS=sum(p.numel() for p in model.parameters())
with torch.no_grad():
    a,b=model(torch.zeros(2,1,WINDOW,device=DEVICE))
assert a.shape==(2,3) and b.shape==(2,2)
print('Parameter count:',f'{PARAMS:,}','FP32 weights MB:',round(PARAMS*4/1e6,2))


## 10. Per-record validation metrics and safety-relevant scoring

A single weighted average is not enough. The notebook measures AF *precision and recall*, Other-rhythm sensitivity, incorrect NORMAL reports, and real-noise rejection. A balanced validation score is only used for ranking candidate research checkpoints; later **hard gates** prevent that score from hiding unacceptable failures.


In [ ]:
def from_logits_to_probs(logits):return torch.softmax(logits.float(),dim=1).cpu().numpy()

@torch.inference_mode()
def evaluate_probs(net,loader,rows):
    net.eval();n=len(rows)
    rs=np.zeros((n,3),dtype=np.float64);qs=np.zeros((n,2),dtype=np.float64)
    qmax=np.zeros(n,dtype=np.float64);counts=np.zeros(n,dtype=np.int32)
    for bx,ids in loader:
        rlog,qlog=net(bx.to(DEVICE,non_blocking=True))
        rp=from_logits_to_probs(rlog);qp=from_logits_to_probs(qlog)
        ix=ids.numpy().astype(int)
        np.add.at(rs,ix,rp);np.add.at(qs,ix,qp)
        np.maximum.at(qmax,ix,qp[:,1]);np.add.at(counts,ix,1)
    assert np.all(counts>0)
    return rs/counts[:,None],qs/counts[:,None],qmax

def score_raw(rows,rp,qp):
    ry=np.asarray([r['ry'] for r in rows]);qy=np.asarray([r['qy'] for r in rows]);
    src=np.asarray([r['source'] for r in rows]);rm=ry>=0;idx17=src=='c2017';idx11=src=='c2011'
    pr=rp.argmax(1);pq=qp.argmax(1)
    prec=precision_score(ry[rm],pr[rm],labels=[0,1,2],average=None,zero_division=0)
    recall=recall_score(ry[rm],pr[rm],labels=[0,1,2],average=None,zero_division=0)
    f1s=f1_score(ry[rm],pr[rm],labels=[0,1,2],average=None,zero_division=0)
    abnormal=rm & np.isin(ry,[1,2]);noisy17=idx17&(qy==1)
    q17ba=float(balanced_accuracy_score(qy[idx17],pq[idx17]))
    q11ba=float(balanced_accuracy_score(qy[idx11],pq[idx11]))
    dangerous=float(np.mean(pr[abnormal]==0)) if np.any(abnormal) else 1.
    noisy_recall=float(np.mean(pq[noisy17]==1)) if noisy17.any() else 0.
    utility=(.23*float(np.mean(f1s))+.18*float(f1s[1])+.15*float(f1s[2])+
             .11*float(recall[1])+.10*float(recall[2])+.12*q17ba+.07*q11ba+.04*(1-dangerous))
    out={'utility':float(utility),'rhythm_macro_f1':float(np.mean(f1s)),
         'rhythm_weighted_f1':float(f1_score(ry[rm],pr[rm],average='weighted',zero_division=0)),
         'per_class_precision':prec.tolist(),'per_class_recall':recall.tolist(),'per_class_f1':f1s.tolist(),
         'af_precision':float(prec[1]),'af_recall':float(recall[1]),
         'other_recall':float(recall[2]),'normal_recall':float(recall[0]),
         'quality_2017_balanced_accuracy':q17ba,'quality_2011_balanced_accuracy':q11ba,
         'quality_2017_noisy_recall':noisy_recall,'abnormal_pred_normal_rate':dangerous}
    out['validation_raw_gate']=(out['rhythm_macro_f1']>=.70 and out['af_precision']>=.62
        and out['af_recall']>=.80 and out['other_recall']>=.52 and out['normal_recall']>=.76
        and out['quality_2017_noisy_recall']>=.50)
    return out

print('Defined raw checkpoint metrics and research validation criteria.')


## 11. Stage 1: rhythm-only warm-up on real 2017 ECGs

No 2011 examples and no signal-quality loss in this stage. Save the best checkpoint based on **rhythm macro F1, AF F1 and Other F1**, not AF sensitivity alone.


In [ ]:
def train_step_rhythm(net,opt,batch):
    x,ry,_=batch;x=x.to(DEVICE);ry=ry.to(DEVICE)
    opt.zero_grad(set_to_none=True)
    logits,_=net(x)
    loss=nn.functional.cross_entropy(logits,ry)
    loss.backward();nn.utils.clip_grad_norm_(net.parameters(),2.0);opt.step()
    return float(loss.detach().cpu())

def save_cpu_state(net):return {k:v.detach().cpu().clone() for k,v in net.state_dict().items()}

warm_opt=torch.optim.AdamW(model.parameters(),lr=LR_WARM,weight_decay=WEIGHT_DECAY)
warm_sched=torch.optim.lr_scheduler.CosineAnnealingLR(warm_opt,T_max=WARM_EPOCHS,eta_min=LR_WARM*.15)
warm_history=[];best_w=-1.;warm_best=None;no_imp=0
start_time=time.time()
for ep in range(1,WARM_EPOCHS+1):
    model.train();losses=[]
    for batch in tqdm(warm_loader,desc=f'Warm-up epoch {ep}/{WARM_EPOCHS}',leave=False):
        losses.append(train_step_rhythm(model,warm_opt,batch))
    warm_sched.step()
    rprob,qprob,_=evaluate_probs(model,val_loader,val_rows)
    keep=np.asarray([r['ry']>=0 for r in val_rows]);ry_true=np.array([r['ry'] for r in val_rows])[keep]
    pred=rprob[keep].argmax(1)
    f1s=f1_score(ry_true,pred,labels=[0,1,2],average=None,zero_division=0)
    value=float(.50*f1s.mean()+.25*f1s[1]+.25*f1s[2])
    warm_history.append({'epoch':ep,'loss':float(np.mean(losses)),'val_rhythm_macro_f1':float(f1s.mean()),
                         'val_f1_normal_af_other':f1s.tolist(),'selection_score':value})
    print(f'Warm-up {ep:02}: loss={np.mean(losses):.4f}, val_macro={f1s.mean():.4f}, '
          f'AF_F1={f1s[1]:.4f}, Other_F1={f1s[2]:.4f}')
    if value>best_w+1e-4:
        best_w=value;warm_best=save_cpu_state(model);no_imp=0
    else:
        no_imp+=1
        if no_imp>=WARM_PATIENCE:print('Warm-up early stopped');break
assert warm_best is not None
print('Warm-up best selection:',round(best_w,4),'elapsed min:',round((time.time()-start_time)/60,2))


## 12. Stage 2: compare two gentle multi-task recipes on validation only

Each recipe starts from **the exact same warm-up checkpoint**. Both are predeclared. Their 2017 and 2011 quality losses are deliberately smaller than the rhythm loss. The auxiliary 2011 loss is low because its labels refer to all 12 leads.

This is a small, controlled hyperparameter comparison—not a guarantee of improvements.


In [ ]:
def get_aux_batch(iterator,loader):
    try:return next(iterator),iterator
    except StopIteration:
        iterator=iter(loader)
        return next(iterator),iterator

def train_joint_epoch(net,opt,loader17,loader11,q2017,q2011,desc):
    net.train();losses=[];iterator11=iter(loader11)
    for x17,ry17,qy17 in tqdm(loader17,desc=desc,leave=False):
        (x11,_,qy11),iterator11=get_aux_batch(iterator11,loader11)
        x17=x17.to(DEVICE);ry17=ry17.to(DEVICE);qy17=qy17.to(DEVICE)
        x11=x11.to(DEVICE);qy11=qy11.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        logits17,quality17=net(x17)
        known=ry17>=0
        l_r=nn.functional.cross_entropy(logits17[known],ry17[known]) if known.any() else 0.0
        l_q17=nn.functional.cross_entropy(quality17,qy17)  # 2017 N/A/O/~ labels
        _,quality11=net(x11)
        l_q11=nn.functional.cross_entropy(quality11,qy11)  # auxiliary noisy 12-lead labels
        loss=l_r+q2017*l_q17+q2011*l_q11
        loss.backward();nn.utils.clip_grad_norm_(net.parameters(),2.0);opt.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

candidate_states={};candidate_best={};joint_histories={}
for idx,recipe in enumerate(RECIPES):
    name=recipe['name'];print('\n'+'='*66+'\nTRAINING RECIPE:',recipe)
    random.seed(SEED+idx+30);np.random.seed(SEED+idx+30);torch.manual_seed(SEED+idx+30)
    net=RhythmQualityNet().to(DEVICE);net.load_state_dict(warm_best)
    l17=weighted_loader(TRAIN_JOINT,JOINT_GROUPS,BATCH17,seed=SEED+100)
    l11=weighted_loader(train11,{'2011_good':.65,'2011_bad':.35},BATCH11,
                        num_samples=max(len(train11),len(TRAIN_JOINT)),seed=SEED+200)
    opt=torch.optim.AdamW(net.parameters(),lr=LR_JOINT,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=JOINT_EPOCHS,eta_min=LR_JOINT*.12)
    best_metric=-1.;best_state=None;no_imp=0;hist=[]
    for ep in range(1,JOINT_EPOCHS+1):
        tl=train_joint_epoch(net,opt,l17,l11,recipe['q2017'],recipe['q2011'],f'{name} {ep}/{JOINT_EPOCHS}')
        sched.step()
        probs_r,probs_q,probs_qmax=evaluate_probs(net,val_loader,val_rows)
        stats=score_raw(val_rows,probs_r,probs_q)
        hist.append({'epoch':ep,'train_loss':tl,**stats})
        print(f'{name} {ep:02}: loss={tl:.4f}, score={stats["utility"]:.4f}, '
              f'macro={stats["rhythm_macro_f1"]:.3f}, AF P/R={stats["af_precision"]:.3f}/{stats["af_recall"]:.3f}, '
              f'OTHER R={stats["other_recall"]:.3f}, noise R={stats["quality_2017_noisy_recall"]:.3f}, '
              f'gate={stats["validation_raw_gate"]}')
        if stats['utility']>best_metric+1e-4:
            best_metric=stats['utility'];best_state=save_cpu_state(net);best_info={'epoch':ep,**stats};no_imp=0
        else:
            no_imp+=1
            if no_imp>=JOINT_PATIENCE:
                print('Early stopping',name);break
    if best_state is None:raise RuntimeError('No trained checkpoint for '+name)
    candidate_states[name]=best_state;candidate_best[name]=best_info;joint_histories[name]=hist
    del net
    if DEVICE.type=='cuda':torch.cuda.empty_cache()
print('\nValidation comparison:')
display(pd.DataFrame([{'recipe':n,**{k:v for k,v in stats.items() if isinstance(v,(float,int,bool))}} for n,stats in candidate_best.items()]).round(4))


## 13. Freeze the validation-selected checkpoint

A recipe meeting the **predeclared** raw metric requirements is preferred. If neither meets them, the notebook selects the better-scoring experimental checkpoint **but reports a failed research gate**. The final within-run test cannot be used to switch models or tune thresholds; previous iterations on the same corpus mean it is not externally independent.


In [ ]:
pass_raw=[n for n,st in candidate_best.items() if st['validation_raw_gate']]
selected_recipe=max(pass_raw if pass_raw else list(candidate_best),
                    key=lambda n:candidate_best[n]['utility'])
model=RhythmQualityNet().to(DEVICE)
model.load_state_dict(candidate_states[selected_recipe]);model.eval()
SELECTED_VAL_RAW=candidate_best[selected_recipe]
print('SELECTED:',selected_recipe,'epoch',SELECTED_VAL_RAW['epoch'])
print('Raw research gate:',bool(SELECTED_VAL_RAW['validation_raw_gate']))
print('Validation metrics:',json.dumps(SELECTED_VAL_RAW,indent=2))
# Audit the frozen weights without exposing raw training signals.
model_sha=hashlib.sha256(b''.join(v.detach().cpu().contiguous().numpy().tobytes()
                                   for v in model.state_dict().values())).hexdigest()
print('Checkpoint SHA-256 fingerprint:',model_sha)


## 14. Choose a conservative rejection policy on validation

Uninterpretable recording probability is aggregated over windows. An optional **normal-class confidence floor** prevents low-confidence abnormal signals being labelled normal. Thresholds are chosen from a *fixed grid* without using any test labels. 2011 test data are never used for selection.


In [ ]:
from itertools import product

def apply_policy(rp,qp,qmax,thresholds):
    rp=np.asarray(rp);qp=np.asarray(qp);qmax=np.asarray(qmax)
    risk=.7*qp[:,1]+.3*qmax
    p=rp.argmax(axis=1).astype(int)
    reject=(risk>=thresholds['max_quality_risk'])|(rp.max(1)<thresholds['min_rhythm_confidence'])
    reject|=((p==0)&(rp[:,0]<thresholds['min_normal_confidence']))
    p[reject]=3
    return p,risk

def policy_stats(rows,pred,risk,quality_threshold):
    source=np.array([r['source'] for r in rows]);ry=np.array([r['ry'] for r in rows]);qy=np.array([r['qy'] for r in rows])
    i17=source=='c2017';i11=source=='c2011'
    y=np.where(qy[i17]==1,3,ry[i17]);p=pred[i17]
    pr=precision_score(y,p,labels=[0,1,2,3],average=None,zero_division=0)
    rc=recall_score(y,p,labels=[0,1,2,3],average=None,zero_division=0)
    cm=confusion_matrix(y,p,labels=[0,1,2,3]);clean=y!=3;abn=np.isin(y,[1,2])
    cov=float(np.mean(p[clean]!=3));false_n=float(np.mean(p[abn]==0))
    # Auxiliary 2011 labels assess 12-lead quality, and are used only for diagnostics.
    q11_score=(np.asarray(risk)[i11]>=quality_threshold).astype(int)
    q11_true=qy[i11]
    q11_ba=float(balanced_accuracy_score(q11_true,q11_score))
    return {'macro_f1':float(f1_score(y,p,labels=[0,1,2,3],average='macro',zero_division=0)),
        'weighted_f1':float(f1_score(y,p,labels=[0,1,2,3],average='weighted',zero_division=0)),
        'per_class_precision':pr.tolist(),'per_class_recall':rc.tolist(),
        'af_precision':float(pr[1]),'af_recall':float(rc[1]),'other_recall':float(rc[2]),
        'normal_recall':float(rc[0]),'noise_rejection_recall':float(rc[3]),
        'clean_coverage':cov,'abnormal_reported_normal_rate':false_n,
        '2011_quality_balanced_accuracy':q11_ba,'confusion_matrix':cm.tolist(),
        'n_2017':len(y),'2017_class_counts':np.bincount(y,minlength=4).tolist(),
        '2011_n':int(i11.sum())}

def policy_utility(st):
    return (0.18*st['macro_f1']+0.18*st['af_recall']+0.15*st['other_recall']+
            0.15*st['noise_rejection_recall']+0.13*st['clean_coverage']+
            0.06*st['af_precision']+0.06*st['per_class_precision'][2]+
            0.09*(1-st['abnormal_reported_normal_rate']))

def gate_policy(st):
    return bool(st['clean_coverage']>=.80 and st['af_recall']>=.74 and
                st['af_precision']>=.62 and st['other_recall']>=.48 and
                st['noise_rejection_recall']>=.55 and st['abnormal_reported_normal_rate']<=.19)

val_rp,val_qp,val_qmax=evaluate_probs(model,val_loader,val_rows)
choices=[]
for qthr,conf,norm in product([.25,.35,.45,.55,.65,.75,.85],[0.,.45,.55,.65],[.0,.55,.70,.80]):
    thresh={'max_quality_risk':float(qthr),'min_rhythm_confidence':float(conf),
            'min_normal_confidence':float(norm)}
    p,risk=apply_policy(val_rp,val_qp,val_qmax,thresh)
    st=policy_stats(val_rows,p,risk,qthr)
    choices.append({'utility':policy_utility(st),'thresholds':thresh,
                    'stats':st,'gate':gate_policy(st)})
passed=[x for x in choices if x['gate']]
# Fall back to non-pathological coverage if all strict gates fail.
base=passed if passed else [x for x in choices if x['stats']['clean_coverage']>=.72]
if not base:base=choices
winner=max(base,key=lambda x:x['utility'])
THRESHOLDS=winner['thresholds'];VAL_POLICY=winner['stats']
VAL_POLICY_GATE_PASSED=bool(passed)
print('Selected thresholds:',THRESHOLDS)
print('Validation research policy gate passed:',VAL_POLICY_GATE_PASSED)
print('Validation policy metrics:\n',json.dumps({k:v for k,v in VAL_POLICY.items() if k!='confusion_matrix'},indent=2))


## 15. FINAL evaluation — within-run held-out test, after decisions are frozen

**Do not use these results to select a new threshold, recipe or checkpoint.** The 2017 corpus was also used in earlier SatellaRhythm experiments; this is a within-run holdout, **not a fresh independent validation cohort**. This cell reports raw rhythm metrics, 2017 portable-ECG policy performance, 2011 auxiliary quality performance, Wilson confidence intervals for small classes, and the **predeclared research acceptance gate**. Failing the gate means *no recommended release*, not “tune on test and rerun”.


In [ ]:
def wilson(k,n,z=1.96):
    if n<1:return [None,None]
    p=k/n;d=1+z*z/n;c=(p+z*z/(2*n))/d
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return [float(max(0,c-h)),float(min(1,c+h))]

test_rp,test_qp,test_qmax=evaluate_probs(model,test_loader,test_rows)
test_pred,test_risk=apply_policy(test_rp,test_qp,test_qmax,THRESHOLDS)
TEST_RAW=score_raw(test_rows,test_rp,test_qp)
TEST_POLICY=policy_stats(test_rows,test_pred,test_risk,THRESHOLDS['max_quality_risk'])
source=np.asarray([r['source'] for r in test_rows]);ry=np.asarray([r['ry'] for r in test_rows]);qy=np.asarray([r['qy'] for r in test_rows])
c17=source=='c2017';c11=source=='c2011'
y17=np.where(qy[c17]==1,3,ry[c17]);p17=test_pred[c17]
print('=== 2017 rhythm-only (N, AF, OTHER), without policy ===')
rm=c17&(ry>=0)
print(classification_report(ry[rm],test_rp[rm].argmax(1),labels=[0,1,2],target_names=RHYTHM,zero_division=0))
print('=== 2017 COMPLETE policy (N, AF, OTHER, UNINTERPRETABLE) ===')
print(classification_report(y17,p17,labels=[0,1,2,3],target_names=RHYTHM+['UNINTERPRETABLE'],zero_division=0))
print('=== 2011 quality (entire 12-lead label transferred to lead I) ===')
q11pred=(test_risk[c11]>=THRESHOLDS['max_quality_risk']).astype(int)
print(classification_report(qy[c11],q11pred,labels=[0,1],target_names=QUALITY,zero_division=0))
print('Confusion: 2017 reference rows, prediction columns [N,AF,OTHER,Q]')
print(np.asarray(TEST_POLICY['confusion_matrix']))
cls_n=np.bincount(y17,minlength=4)
for index,tag in [(1,'AF recall'),(2,'Other recall'),(3,'Real noisy rejection')]:
    succ=int(np.sum((y17==index)&(p17==index)));n=int(cls_n[index])
    print(f'{tag}: {succ}/{n} = {succ/max(n,1):.3f}; approximate 95% Wilson CI:',wilson(succ,n))
abn=np.isin(y17,[1,2]);unsafe=int(np.sum(abn&(p17==0)));total=int(np.sum(abn))
print(f'ABNORMAL reported NORMAL: {unsafe}/{total}; CI:',wilson(unsafe,total))

RESEARCH_TEST_GATES={
   'four_way_macro_f1_ge_0_72':TEST_POLICY['macro_f1']>=.72,
   'af_recall_ge_0_80':TEST_POLICY['af_recall']>=.80,
   'af_precision_ge_0_65':TEST_POLICY['af_precision']>=.65,
   'other_recall_ge_0_55':TEST_POLICY['other_recall']>=.55,
   'noise_recall_ge_0_60':TEST_POLICY['noise_rejection_recall']>=.60,
   'clean_coverage_ge_0_80':TEST_POLICY['clean_coverage']>=.80,
   'abnormal_reported_normal_le_0_15':TEST_POLICY['abnormal_reported_normal_rate']<=.15,
}
RESEARCH_GATE_PASS=bool(all(RESEARCH_TEST_GATES.values()) and
                        SELECTED_VAL_RAW['validation_raw_gate'] and VAL_POLICY_GATE_PASSED)
print('\nRESEARCH RELEASE-CANDIDATE GATE:', 'PASS' if RESEARCH_GATE_PASS else 'NOT MET')
for k,v in RESEARCH_TEST_GATES.items():print(('PASS' if v else 'FAIL'),k)
print('NOT a clinical validation or medical-device safety gate.')

fig,ax=plt.subplots(figsize=(7,5));cm=np.asarray(TEST_POLICY['confusion_matrix'])
ax.imshow(cm,cmap='Blues')
for i in range(4):
    for j in range(4):ax.text(j,i,str(cm[i,j]),ha='center',va='center')
ax.set_xticks(range(4),['Normal','AF','Other','Uninterpretable'],rotation=20,ha='right')
ax.set_yticks(range(4),['Normal','AF','Other','Uninterpretable'])
ax.set(xlabel='Predicted',ylabel='Reference',title='SatellaRhythm v0.3 — held-out 2017 ECGs')
plt.tight_layout();plt.show()


## 16. Research inference on a portable ECG waveform

Input must be a **single-lead ECG waveform**, not PPG/pulse data. A low-quality or low-confidence recording may be returned as `REVIEW_UNINTERPRETABLE`. A predicted `NORMAL` never means the person has been medically cleared.

Actual edge support depends on device-specific ADC calibration, lead placement, preprocessing, performance and independent clinical evaluation.


In [ ]:
@torch.inference_mode()
def predict_ecg_research(ecg,fs=300,n_windows=5,net=None):
    net=model if net is None else net
    if not isinstance(fs,(float,int)) or fs<=0 or fs>2000:
        return {'status':'UNSUPPORTED_SAMPLE_RATE','research_only':True}
    try:x=np.asarray(ecg,dtype=np.float32).reshape(-1)
    except Exception:return {'status':'INVALID_INPUT','research_only':True}
    if not np.isfinite(x).all() or len(x)<int(8.5*fs) or len(x)>int(120*fs) or np.std(x)<1e-5:
        return {'status':'INVALID_OR_INSUFFICIENT_SIGNAL','research_only':True}
    if int(fs)!=FS:
        from fractions import Fraction
        ratio=Fraction(FS,1)/Fraction(str(fs)).limit_denominator(10000)
        x=sps.resample_poly(x,ratio.numerator,ratio.denominator).astype(np.float32)
    starts=np.linspace(0,max(0,len(x)-WINDOW),n_windows,dtype=int)
    bx=np.stack([preprocess(crop(x,int(s))) for s in starts],0)[:,None,:]
    pred_r,pred_q=net(torch.from_numpy(bx).to(next(net.parameters()).device))
    rp=from_logits_to_probs(pred_r).mean(0)[None,:]
    qwin=from_logits_to_probs(pred_q)
    qp=qwin.mean(0)[None,:];qmax=np.array([qwin[:,1].max()])
    decision,risk=apply_policy(rp,qp,qmax,THRESHOLDS)
    label=(RHYTHM+['REVIEW_UNINTERPRETABLE'])[int(decision[0])]
    return {'status':'EXPERIMENTAL_RESEARCH_ONLY','prediction':label,
      'rhythm_probabilities':dict(zip(RHYTHM,rp[0].round(4).tolist())),
      'quality_risk':round(float(risk[0]),4),
      'not_for_diagnosis_or_reassurance':True}

example=next((r for r in test17 if r['source_label']=='A'),None)
if example is not None:
    print('Held-out sample example:',predict_ecg_research(example['x']))
    plt.figure(figsize=(11,2));plt.plot(np.arange(WINDOW)/FS,crop(example['x'],0),lw=.7)
    plt.title('Example single-lead ECG window from reference dataset');plt.xlabel('Time (s)');plt.tight_layout();plt.show()


## 17. Save weights, validated ONNX and reproducibility metadata

The export is **CPU-capable** and has **two separate outputs**. The exact preprocessing and threshold policy are stored in `config_research.json`. The repository package contains *no raw ECGs, user data, Colab logs or credentials*.


In [ ]:
import onnx, onnxruntime as ort
from safetensors.torch import save_file
from dataclasses import asdict

cpu_net=RhythmQualityNet().cpu()
cpu_net.load_state_dict(save_cpu_state(model));cpu_net.eval()
weights={k:v.contiguous().cpu() for k,v in cpu_net.state_dict().items()}
save_file(weights,str(MODEL_DIR/'model.safetensors'))
onnx_path=MODEL_DIR/'satellarhythm_edge_v03.onnx'
dummy=torch.randn(2,1,WINDOW)
with torch.inference_mode():ref=cpu_net(dummy)
# Legacy exporter is chosen for broad Colab/PyTorch compatibility. The two outputs are explicit.
torch.onnx.export(cpu_net,dummy,str(onnx_path),opset_version=17,dynamo=False,
                  input_names=['ecg'],output_names=['rhythm_logits','quality_logits'],
                  dynamic_axes={'ecg':{0:'batch'},'rhythm_logits':{0:'batch'},'quality_logits':{0:'batch'}})
onnx.checker.check_model(str(onnx_path))
ort_session=ort.InferenceSession(str(onnx_path),providers=['CPUExecutionProvider'])
onnx_outputs=ort_session.run(None,{'ecg':dummy.numpy()})
PARITY=max(float(np.max(np.abs(a.detach().numpy()-b))) for a,b in zip(ref,onnx_outputs))
assert PARITY<1e-3,f'ONNX mismatch: max abs error {PARITY}'
print('ONNX parity max absolute logit difference:',PARITY)
CONFIG={
  'name':'SatellaRhythm-Edge v0.3','architecture':'dual_head_1d_resnet_mean_max_pool',
  'parameters':PARAMS,'sampling_rate_hz':FS,'window_seconds':SECONDS,'window_samples':WINDOW,
  'preprocessing':'subtract median, divide by max(90th percentile abs amplitude,1e-6), clip [-6,6]',
  'inference_windows':5,'rhythm_classes':RHYTHM,'quality_classes':QUALITY,
  'quality_risk':'0.7*mean quality class-1 probability + 0.3*max class-1 probability',
  'thresholds':THRESHOLDS,'selected_recipe':selected_recipe,
  'training_datasets':['Challenge 2017 v3; portable single-lead ECG','Challenge 2011 Set A; 12-lead quality labels applied to lead I']+
                      (['AFDB; AFIB training windows only'] if ENABLE_AFDB else []),
  'no_synthetic_noise':True,'patient_disjoint_verified':False,
  'external_device_tested':False,'medical_device_approved':False,'research_only':True,
  'weights_sha256':hashlib.sha256((MODEL_DIR/'model.safetensors').read_bytes()).hexdigest(),
}
METRICS={
   'configuration':{'warm_epochs':WARM_EPOCHS,'joint_epochs':JOINT_EPOCHS,'lr_warm':LR_WARM,'lr_joint':LR_JOINT,
      'recipes':RECIPES,'seed':SEED,'afdb_enabled':ENABLE_AFDB},
   'split_audit':SPLIT_AUDIT,
   'warm_best_score':best_w,'warm_history':warm_history,
   'candidate_validation_best':candidate_best,'joint_histories':joint_histories,
   'selected_recipe':selected_recipe,'selected_validation_raw_metrics':SELECTED_VAL_RAW,
   'selected_thresholds':THRESHOLDS,'validation_policy':VAL_POLICY,
   'test_raw_metrics':TEST_RAW,'test_policy':TEST_POLICY,
   'test_research_gates':RESEARCH_TEST_GATES,
   'research_gate_pass':RESEARCH_GATE_PASS,
   'onnx_parity_max_abs_error':PARITY,
   'limitations':['2017/2011 patient-disjoint status unknown','2011 quality labels from entire 12-lead record',
                  'same-device 2017 split is not independent-device validation','research only; no clinical safety demonstrated']}
(MODEL_DIR/'config_research.json').write_text(json.dumps(CONFIG,indent=2)+'\n',encoding='utf-8')
(MODEL_DIR/'metrics.json').write_text(json.dumps(METRICS,indent=2)+'\n',encoding='utf-8')
print('Saved weights / ONNX / structured metrics')


## 18. Edge CPU benchmark

Measures the current Colab CPU, **not** an actual wearable or mobile SoC. Quantisation is intentionally left out of the first export so the recorded parity is for the actual released FP32 model.


In [ ]:
import statistics
cpu_input=np.random.default_rng(SEED).normal(0,1,(1,1,WINDOW)).astype('float32')
batch_input=np.repeat(cpu_input,5,axis=0)
for _ in range(8):ort_session.run(None,{'ecg':cpu_input})
def latency(x,n=40):
    ts=[]
    for _ in range(n):
        t=time.perf_counter();ort_session.run(None,{'ecg':x});ts.append((time.perf_counter()-t)*1000)
    return {'median_ms':float(np.median(ts)),'p95_ms':float(np.percentile(ts,95))}
LAT_SINGLE=latency(cpu_input);LAT_FIVE=latency(batch_input)
METRICS['cpu_onnx_single_window']=LAT_SINGLE;METRICS['cpu_onnx_five_window_batch']=LAT_FIVE
(MODEL_DIR/'metrics.json').write_text(json.dumps(METRICS,indent=2)+'\n',encoding='utf-8')
print('Colab CPU, 10s window:',LAT_SINGLE,'5-window batch:',LAT_FIVE)


## 19. Standalone ONNX inference module and detailed public-facing research model card

The package includes one complete `infer_onnx.py` that can be copied outside Colab and used with NumPy, SciPy and ONNX Runtime. It has **no network calls** and always indicates research-only status. The README is generated from the measured results, not invented figures.


In [ ]:
infer_script=r'''# SatellaRhythm-Edge v0.3 — experimental ONNX single-lead ECG inference.
# Not for diagnosis, screening, treatment, triage or health reassurance.
# Requires numpy, scipy, onnxruntime. Usage: from infer_onnx import RhythmEdge
import json
from pathlib import Path
from fractions import Fraction
import numpy as np
from scipy.signal import resample_poly
import onnxruntime as ort

class RhythmEdge:
    def __init__(self, folder):
        folder=Path(folder)
        self.cfg=json.loads((folder/'config_research.json').read_text(encoding='utf-8'))
        self.session=ort.InferenceSession(str(folder/'satellarhythm_edge_v03.onnx'),providers=['CPUExecutionProvider'])
        self.fs=int(self.cfg['sampling_rate_hz']);self.win=int(self.cfg['window_samples'])
        self.thr=self.cfg['thresholds']
    def _window(self,x,start):
        y=np.asarray(x[start:start+self.win],dtype=np.float32)
        if y.size<self.win:y=np.pad(y,(0,self.win-y.size),mode='reflect')
        y=y-np.median(y)
        scale=max(float(np.percentile(np.abs(y),90)),1e-6)
        return np.clip(y/scale,-6,6).astype(np.float32)
    @staticmethod
    def _softmax(a):
        a=a-a.max(axis=-1,keepdims=True)
        e=np.exp(a);return e/e.sum(axis=-1,keepdims=True)
    def predict(self,ecg,sample_rate_hz=300,n_windows=5):
        fs=float(sample_rate_hz)
        if fs<=0 or fs>2000:return {'status':'UNSUPPORTED_SAMPLE_RATE','research_only':True}
        try:x=np.asarray(ecg,dtype=np.float32).reshape(-1)
        except Exception:return {'status':'INVALID_INPUT','research_only':True}
        if not np.isfinite(x).all() or len(x)<int(8.5*fs) or len(x)>int(120*fs) or np.std(x)<1e-5:
            return {'status':'INVALID_OR_INSUFFICIENT_SIGNAL','research_only':True}
        if fs!=self.fs:
            rat=Fraction(self.fs,1)/Fraction(str(fs)).limit_denominator(10000)
            x=resample_poly(x,rat.numerator,rat.denominator).astype(np.float32)
        starts=np.linspace(0,max(0,len(x)-self.win),int(n_windows),dtype=int)
        inp=np.stack([self._window(x,int(s)) for s in starts])[:,None,:]
        rl,ql=self.session.run(None,{'ecg':inp})
        rp=self._softmax(rl).mean(0);qs=self._softmax(ql)
        risk=.7*float(qs[:,1].mean())+.3*float(qs[:,1].max())
        label=int(rp.argmax());low_conf=float(rp.max())<self.thr['min_rhythm_confidence']
        if label==0 and float(rp[0])<self.thr['min_normal_confidence']:low_conf=True
        if risk>=self.thr['max_quality_risk'] or low_conf:label=3
        names=self.cfg['rhythm_classes']+['REVIEW_UNINTERPRETABLE']
        return {'status':'EXPERIMENTAL_RESEARCH_ONLY','prediction':names[label],
            'rhythm_probabilities':dict(zip(self.cfg['rhythm_classes'],rp.astype(float).round(4).tolist())),
            'quality_risk':round(risk,4),'not_for_diagnosis_or_reassurance':True}
'''
compile(infer_script,'infer_onnx.py','exec')
(MODEL_DIR/'infer_onnx.py').write_text(infer_script,encoding='utf-8')

cm=np.asarray(TEST_POLICY['confusion_matrix'])
card=f'''---
tags:
- ecg
- single-lead-ecg
- onnx
- physiological-signal-processing
- time-series-classification
- research
metrics:
- f1
---
# SatellaRhythm-Edge v0.3 — single-lead ECG research release candidate

> **Research only. Not a diagnostic, screening, triage or medical device.** No result, including `NORMAL`, should be used to reassure a patient or decide whether medical care is needed. No independent clinical validation has been performed.

## Summary

Small **{PARAMS:,}-parameter** 1D residual CNN trained from scratch on real ECG recordings. It accepts a **10-second, single-lead ECG waveform sampled at 300 Hz** and returns two logits arrays: rhythm (`NORMAL`, `ATRIAL_FIBRILLATION`, `OTHER_RHYTHM`) and signal quality (`INTERPRETABLE`, `UNINTERPRETABLE`). A validation-selected abstention policy reports `REVIEW_UNINTERPRETABLE` if the quality risk or confidence is inadequate. No optical pulse/PPG support.

The model uses mean/max temporal pooling, rhythm-first warm-up, then two alternate low-weight multi-task recipes selected solely by validation. Synthetic noisy recordings are **not** part of training.

## Transparent research status

**Experimental research test gate: {'PASSED' if RESEARCH_GATE_PASS else 'NOT MET'}**. This is an internal engineering decision aid, **not** a medical safety or regulatory threshold. {'The checkpoint is still not clinically validated.' if RESEARCH_GATE_PASS else 'The benchmark goals were not reached. Do not describe this checkpoint as production-ready.'}

## Training sources and limitations

1. [PhysioNet/CinC Challenge 2017](https://physionet.org/content/challenge-2017/1.0.0/), 300 Hz portable AliveCor single-lead ECGs, **v3** revised labels N/A/O/~. All were used strictly as rhythm-labelled or quality-labelled according to their provided annotations. `~` is not given an invented rhythm label. A label N/A/O is treated as interpretable for this experiment but may contain noise.
2. [PhysioNet/CinC Challenge 2011 Set A](https://physionet.org/content/challenge-2011/1.0.0/), 500 Hz 12-lead ECGs. Lead I is extracted and resampled to 300 Hz. **The quality label was assigned to the entire 12-lead record**, not specifically to lead I; this is noisy auxiliary supervision. No rhythm labels are imputed.
3. Optional [MIT-BIH AFDB](https://physionet.org/content/afdb/1.0.0/) annotated AF windows: **{'enabled' if ENABLE_AFDB else 'disabled'}**; if enabled, used in training only.

Only exact waveform duplicates were removed. Training/validation/test are **record-disjoint within this experiment**, not confirmed patient-disjoint; earlier SatellaRhythm versions already inspected this source dataset, so the reported test is not a newly independent cohort. PhysioNet 2017 test results are not validation on an entirely different wearable manufacturer or patient population. Review the original datasets' terms and citation instructions before any redistribution or commercial use; no unrestricted commercial licence for derivative weights is asserted here.

## Architecture and preprocessing

- 1D ResNet with residual Conv1D blocks, shared temporal mean/max pooling, separate 3-way rhythm and 2-way quality heads.
- {PARAMS:,} parameters. Model weights: `model.safetensors`; CPU ONNX: `satellarhythm_edge_v03.onnx`.
- Input window: {WINDOW} samples ({SECONDS} s at {FS} Hz). `float32` shape `(batch,1,{WINDOW})`.
- Per-window median subtraction, 90th-percentile absolute residual scaling, clipping to [-6,6]. Resample non-300Hz ECGs before windowing; confirm physical lead compatibility.
- Multi-window recording aggregation: mean softmax probabilities; quality risk = 0.7 mean plus 0.3 maximum uninterpretable probability.
- Selected recipe: `{selected_recipe}`. Selection checkpoint validation score: {SELECTED_VAL_RAW['utility']:.4f}.
- Validation rejection policy: `{json.dumps(THRESHOLDS)}`.

## Evaluation — original held-out recording partitions

### Rhythm classification before abstention (2017 clean rhythm-labelled records)

- Macro F1: **{TEST_RAW['rhythm_macro_f1']:.4f}**.
- Weighted F1: **{TEST_RAW['rhythm_weighted_f1']:.4f}** (heavily affected by class frequencies).
- Per-class recall (N, AF, Other): `{[round(v,4) for v in TEST_RAW['per_class_recall']]}`.
- AF precision: **{TEST_RAW['af_precision']:.4f}**.

### Four-class output after abstention (2017)

| Metric | Value |
|---|---:|
| Four-class macro F1 | **{TEST_POLICY['macro_f1']:.4f}** |
| AF precision | **{TEST_POLICY['af_precision']:.4f}** |
| AF sensitivity | **{TEST_POLICY['af_recall']:.4f}** |
| Other-rhythm sensitivity | **{TEST_POLICY['other_recall']:.4f}** |
| Uninterpretable recording recall | **{TEST_POLICY['noise_rejection_recall']:.4f}** |
| Interpretable-record coverage | **{TEST_POLICY['clean_coverage']:.4f}** |
| AF/Other incorrectly reported NORMAL | **{TEST_POLICY['abnormal_reported_normal_rate']:.4f}** |
| Auxiliary 2011 whole-record quality balanced accuracy applied to lead I | **{TEST_POLICY['2011_quality_balanced_accuracy']:.4f}** |

2017 confusion matrix (reference rows, predictions columns N/AF/Other/Q):

```
{cm.tolist()}
```

Very small `~` counts make quality recall estimates uncertain. See `metrics.json` for exact n, class counts, validation results, source-specific results, and predefined gate outcomes. These are research scores, not comparable to published clinical sensitivities or an official PhysioNet leaderboard entry.

### Inference speed and agreement

- ONNX/PyTorch maximum tested absolute logit difference: **{PARITY:.3g}**.
- ONNX Runtime CPU median on Colab (single 10-second window): **{LAT_SINGLE['median_ms']:.3f} ms**; batch of five windows **{LAT_FIVE['median_ms']:.3f} ms**. Hardware-dependent; no real phone/watch benchmark.

## Easy Python/ONNX usage

```bash
pip install numpy scipy onnxruntime
```

Copy `infer_onnx.py`, `config_research.json` and `satellarhythm_edge_v03.onnx` into a folder:

```python
import numpy as np
from infer_onnx import RhythmEdge
model = RhythmEdge('./model')
# ecg = np.asarray(<YOUR SINGLE-LEAD ECG WAVEFORM>, dtype=np.float32)
# result = model.predict(ecg, sample_rate_hz=300)
# print(result)
```

The model **does not** accept JPEG/PNG ECG images, Bluetooth pulse readings, PPG/optical heart rate, or R-peaks alone. The example intentionally does not invent or bundle medical recordings. Use only real ECG with an appropriate legal basis and privacy protections. Caller is responsible for checking acquisition, sampling and lead placement.

## Known failure modes

- Can miss atrial fibrillation or other abnormal rhythms, or wrongly classify them as normal. This may be harmful if used for reassurance.
- May overcall AF; AF probability is not a clinical diagnosis. Other-rhythm class is highly heterogeneous.
- Signal quality differs by lead/device; original 2011 labels assess complete 12-lead signals rather than lead I.
- Not validated on a new consumer-device manufacturer, patient-disjoint cohort, paediatric cohort, demographic subgroups, noisy real-world wearing conditions, or clinical workflow.
- Threshold selection and internal research gates do not constitute clinical safety testing or regulatory approval.
- Requires additional independent evaluation, expert review, prospective device validation, and applicable regulatory assessment before any clinical deployment.

## Data and attribution

Cite PhysioNet, Challenge 2017 and Challenge 2011 and their original authors using the dataset pages above. Please examine each source's licence and original attribution/citation requirements; the model release card does **not** grant rights over source data. No raw ECGs or patient records are bundled.
'''
(MODEL_DIR/'README.md').write_text(card,encoding='utf-8')
print('Generated README and infer_onnx.py. Research gate:',RESEARCH_GATE_PASS)


## 20. Strict model-only packaging, integrity checks and downloads

Only allowlisted model files are zipped. The ZIP excludes training archives, datasets, logs, notebooks and hidden tokens. Generated metadata contains no creator identity or local paths. Upload to Hugging Face is deliberately **not** automatic; if research gates fail, the package remains for investigation only.


In [ ]:
from zipfile import ZipFile,ZIP_DEFLATED
allowed={'model.safetensors','satellarhythm_edge_v03.onnx','config_research.json','metrics.json','README.md','infer_onnx.py'}
actual={p.name for p in MODEL_DIR.iterdir() if p.is_file()}
assert actual==allowed, f'Unexpected model files; inspect before packaging: {actual^allowed}'
for name in ('README.md','infer_onnx.py','config_research.json','metrics.json'):
    txt=(MODEL_DIR/name).read_text(encoding='utf-8')
    assert not re.search(r'(?:/content/drive/|/home/|hf_[A-Za-z0-9]{12,}|ghp_[A-Za-z0-9]{12,}|Bearer\s+[A-Za-z0-9_.-]+)',txt), f'Possible private metadata in {name}'
assert json.loads((MODEL_DIR/'config_research.json').read_text())['research_only']
package=ROOT/'SatellaRhythm_Edge_v03_Research_Model.zip'
with ZipFile(package,'w',ZIP_DEFLATED) as z:
    for name in sorted(allowed):z.write(MODEL_DIR/name,arcname=name)
with ZipFile(package) as z:
    assert set(z.namelist())==allowed
    assert z.testzip() is None
print('PACKAGE READY:',package,'size MB:',round(package.stat().st_size/1e6,2))
print('Model checkpoint research release gate:',RESEARCH_GATE_PASS)
print('No automatic HF upload. No protected patient waveform included. Do NOT claim clinical safety.')
try:
    from google.colab import files as colab_files
    print('To download: Colab Files pane → /content/satellarhythm_edge_v03/')
    # Uncomment ONLY if desired: colab_files.download(str(package))
except ImportError:pass


## 21. How to decide whether this is the final research checkpoint

Review **every** predeclared test gate. This corpus was also used in earlier SatellaRhythm experiments, so these results must not be presented as independent external confirmation. Even a green research result cannot establish clinical suitability. If any gate fails, archive the checkpoint and conduct a **new development study with a fresh untouched test cohort**; do not repeatedly revise against the same test set.

**Most important measures:** AF false negatives, false-normal abnormal rhythms, Other-rhythm recall, real noise rejection, interpretable ECG coverage, and 2011 quality transfer. In production, positive/negative predictive value depends on patient prevalence, which our curated data does not represent.

For any clinical or consumer health use, independent multi-device, patient-disjoint, prospectively collected data, clinician review, and applicable medical-device regulations are indispensable. No “final” Colab notebook can substitute for those steps.
